# Prédiction des Champions NBA 2026-2030
## Modélisation avancée sur 75 ans d'histoire des franchises

**Objectif :** Déterminer quelles franchises ont le plus de chances de gagner au moins un titre dans les 5 prochaines saisons (2026–2030).

**Modèles utilisés :** Random Forest, XGBoost, LightGBM, Voting Classifier

**Validation :** Leave-One-Out (LOO)

**Interprétation :** SHAP Values

In [19]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')


df = pd.read_csv("C:\\Users\\robba\\OneDrive\\Bureau\\Python_Project_M1_BDIA\\data\\features\\nba_teams_features.csv")
print(f"{len(df)} franchises chargées")

43 franchises chargées


In [20]:
current_teams = [
    'BOS','LAL','SAS','OKC','PHO','UTA','MIL','MIA','POR','DEN','HOU','PHI','IND',
    'CHI','DAL','ATL','NYK','GSW','CLE','DET','TOR','ORL','NOP','SAC',
    'BKN','MEM','WAS','LAC','MIN','CHA'
]

df = df[df['Abbreviation'].isin(current_teams)].copy()


In [21]:
# ------------------------------------------------------
# 2) CIBLE : a gagné un titre dans l’ère moderne (2000+)
# ------------------------------------------------------
modern_champ = ['LAL','SAS','GSW','MIA','CLE','BOS','DAL','TOR','MIL','DEN']

df['Target'] = df['Abbreviation'].isin(modern_champ).astype(int)


In [22]:
# ------------------------------------------------------
# 3) Features (sans fuite—aucune stat future)
# ------------------------------------------------------
df['Dynasty_Power'] = (
    df['AllTime_WinRate'] * 0.40 +
    df['Playoff_Rate'] * 0.30 +
    df['Champ_Rate'] * 0.20 +
    df['Finals_Rate'] * 0.10
)

features = ['AllTime_WinRate','Playoff_Rate','Champ_Rate','Finals_Rate',
            'Dynasty_Power','Age','Champ','Has_Relocated','ABA_Origin']

X = df[features]
y = df['Target']


In [23]:
# ------------------------------------------------------
# 4) Comparaison des modèles avec LOOCV (méthode PRO)
# ------------------------------------------------------
models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=600,
        max_depth=7,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        verbose=-1
    )
}

loo = LeaveOneOut()
results = {}

print("\n================ COMPARAISON LOOCV ================\n")

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=loo, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name:12} -> Accuracy = {scores.mean():.3f}  Std = {scores.std():.3f}")



================ COMPARAISON LOOCV ================

RandomForest -> Accuracy = 0.593  Std = 0.491
XGBoost      -> Accuracy = 0.593  Std = 0.491
LightGBM     -> Accuracy = 0.630  Std = 0.483


In [24]:
best_name = max(results, key=results.get)
best_model = models[best_name]

print("\n================ MEILLEUR MODELE ================\n")
print(f"Modele selectionne : {best_name}  Accuracy = {results[best_name]:.3f}")



best_model.fit(X, y)



================ MEILLEUR MODELE ================

Modele selectionne : LightGBM  Accuracy = 0.630


LGBMClassifier(learning_rate=0.05, max_depth=5, n_estimators=600, verbose=-1)

In [25]:
df['Win_Probability_2026'] = best_model.predict_proba(X)[:, 1]

df_sorted = df.sort_values('Win_Probability_2026', ascending=False)

champ_team = df_sorted.iloc[0]['Team']
champ_abbr = df_sorted.iloc[0]['Abbreviation']
champ_proba = df_sorted.iloc[0]['Win_Probability_2026']

print("\n================ CHAMPION NBA 2025-2026 ================\n")
print(f"Champion predit : {champ_team} ({champ_abbr})")
print(f"Probabilite : {champ_proba:.3f}")



================ CHAMPION NBA 2025-2026 ================

Champion predit : Boston Celtics (BOS)
Probabilite : 0.370


In [26]:
os.makedirs("notebooks", exist_ok=True)

joblib.dump(best_model, "best_champion_model.joblib")

print("Modele sauvegarde")


Modele sauvegarde
